In [43]:
import ir_datasets
import pandas as pd

# dl19 = ir_datasets.load("msmarco-document/trec-dl-2019/judged")
# dataset = ir_datasets.load("msmarco-passage/trec-dl-2019/judged")
dataset = ir_datasets.load("msmarco-passage/trec-dl-2020/judged")

In [44]:
qrels = pd.DataFrame(dataset.qrels)
queries = pd.DataFrame(dataset.queries)

In [45]:
qrels = qrels.merge(queries, left_on='query_id', right_on='query_id')

In [46]:
orcas = ir_datasets.load("msmarco-document/orcas")

In [47]:
orcas_qrels = pd.DataFrame(orcas.qrels)

In [48]:
orca_queries = pd.DataFrame(orcas.queries)

In [49]:
orcas_qrels["doc_id"] = orcas_qrels["doc_id"].str.strip("D")

In [50]:
qrels_in_dl = orcas_qrels[orcas_qrels['doc_id'].isin(set(qrels["doc_id"]))]

In [51]:
candidate_queries = qrels_in_dl.merge(
    orca_queries, left_on='query_id', right_on='query_id')

In [52]:
import os

In [2]:
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

In [3]:
import torch
from sentence_transformers import SentenceTransformer


In [4]:
model_name: str = "Qwen/Qwen3-Embedding-8B"
model = SentenceTransformer(model_name)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [14]:
def encode_texts(texts, model, batch_size=32):
    ret = []
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]
        with torch.no_grad():
            embeddings = model.encode(batch_texts, convert_to_tensor=True)

        ret.extend(embeddings.cpu().numpy())
    return ret       


In [54]:
query_embeddings = encode_texts(queries["text"].to_list(), model)

In [55]:
candidate_embeddings = encode_texts(candidate_queries["text"].to_list(), model)

In [31]:
def knn_search(query_embeddings, variant_embeddings, k=10):
    import numpy as np
    from sklearn.neighbors import NearestNeighbors

    nbrs = NearestNeighbors(n_neighbors=k, algorithm='auto',
                            metric='cosine').fit(variant_embeddings)
    distances, indices = nbrs.kneighbors(query_embeddings)
    return distances, indices

In [57]:
distances, indices = knn_search(query_embeddings, candidate_embeddings, k=10)

In [58]:
query_id = 3
print("Query:", queries.iloc[query_id]["text"])
candidate_queries.iloc[indices[query_id]]

Query: who sings monk theme song


,query_id,doc_id,relevance,iteration,text
20446,3356338,3477583,1,0,the monks
22978,4789692,3477583,1,0,what is a buddhist monk
14196,2361275,3477583,1,0,monks in thailand
18708,8114031,2668726,1,0,song of music
20329,9592462,3477583,1,0,thailand monks
14414,2206102,2668726,1,0,music from the sound of music
20499,10820213,2668726,1,0,the sound of music songs
22793,10049490,3477583,1,0,what do buddhist monks do
6148,5019329,3477583,1,0,daily life of a buddhist monk
20260,6966744,3113521,1,0,tenor singer


In [ ]:
query_id = 2
print("Query:", queries.iloc[query_id]["text"])
candidate_queries.iloc[indices[query_id]]

Query: why did the us volunterilay enter ww1


,query_id,doc_id,relevance,iteration,text
19361,6463896,345585,1,0,us navy history
8823,11757818,345585,1,0,history of us navy
19357,9879074,345585,1,0,us navy established
21127,4230680,345585,1,0,when was the us navy established
21126,4838280,345585,1,0,when was the us navy created
17,4748928,465438,1,0,1 world trade
21531,6303845,465438,1,0,wtc1
18510,9049960,345585,1,0,the us navy
19372,3162950,345585,1,0,us navy wiki
21536,7898075,120982,1,0,ww.ssa.gov
